# 额外的周末练习 - 第 2 周

现在，使用您从第 2 周学到的所有知识为您在第 1 周练习中构建的技术问题/回答器构建完整的原型。

这应该包括 Gradio UI、流媒体、使用系统提示来添加专业知识以及在模型之间切换的能力。如果您能够演示工具的使用，则可获得奖励积分！

如果您觉得大胆，请看看是否可以添加音频输入，以便您可以与它交谈，并让它用音频进行响应。 ChatGPT 或 Claude 可以帮助您，如果您有疑问，也可以给我发电子邮件。

我很快就会在这里发布完整的解决方案 - 除非有人比我先一步......

这方面的商业应用有很多，从语言导师到公司入职解决方案，再到人工智能伴侣和课程（就像这个！），我迫不及待地想看到你的结果。

In [ ]:
# 导入
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_contents

In [ ]:
# 初始化: load env, create client, model options (OpenAI only - switch for speed vs quality)
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()
MODELS = {"gpt-4.1-mini": "gpt-4.1-mini", "gpt-5-nano (faster/cheaper)": "gpt-5-nano"}
current_model = ["gpt-4.1-mini"]  # default (dropdown label); updated by dropdown, mapped to id in chat()

In [ ]:
# 系统提示：技术导师专业知识（Python、软件工程、数据科学、法学硕士）
system_message = """You are a helpful technical tutor. You answer questions about Python, software engineering, data science, and LLMs in a clear, concise way.
Give accurate explanations. If you don't know something, say so. Use markdown for structure (headers, lists, code snippets) when helpful.
When the user shares a URL, you may use the fetch_url_content tool to get the page content and then explain or summarize it. Use the tool when it would help answer their question."""

In [ ]:
# 工具：获取 URL 内容（以便助手可以阅读页面并解释/总结它）
def fetch_url_content(url: str) -> str:
    """Fetch and return the text content of a webpage. Use when the user asks about a URL."""
    try:
        return fetch_website_contents(url)
    except Exception as e:
        return f"Error fetching URL: {e}"

fetch_url_function = {
    "name": "fetch_url_content",
    "description": "Fetch the text content of a webpage at the given URL. Use when the user asks to explain, summarize, or read a web page.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "Full URL of the page (e.g. https://example.com)"},
        },
        "required": ["url"],
        "additionalProperties": False,
    },
}
tools = [{"type": "function", "function": fetch_url_function}]

In [ ]:
# 处理来自模型的工具调用（运行我们的函数并返回工具结果）
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "fetch_url_content":
            args = json.loads(tool_call.function.arguments)
            url = args.get("url", "")
            content = fetch_url_content(url)
            responses.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            })
    return responses

In [ ]:
# 流式聊天：对话历史记录+工具支持（循环直到模型停止请求工具）
def chat(message, history):
    model = MODELS.get(current_model[0], current_model[0])  # map dropdown label to API model id
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    # 在循环中处理工具调用（模型可能会调用工具，我们将结果发送回，它可能会再次调用）
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_responses = handle_tool_calls(msg)
        messages.append(msg)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    # 流式传输最终文本回复（打字机效果）：逐字生成
    final_content = response.choices[0].message.content or ""
    if not final_content:
        yield "(No text reply)"
        return
    result = ""
    for word in final_content.split():
        result += word + " "
        yield result

In [ ]:
# Gradio 界面: model dropdown + chat interface (流式输出, markdown, examples)
def set_model(m):
    current_model[0] = m
    return m

with gr.Blocks(title="Technical Q&A Tutor", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Week 2 – Technical Q&A Tutor\nAsk anything about Python, software engineering, data science, or LLMs. Paste a URL and ask to explain or summarize it—the assistant can fetch the page.")
    model_selector = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="gpt-4.1-mini",
        label="Model",
        info="Switch between faster/cheaper (gpt-5-nano) and stronger (gpt-4.1-mini)",
    )
    model_selector.change(fn=set_model, inputs=model_selector)
    gr.ChatInterface(
        fn=chat,
        type="messages",
        examples=[
            "Explain what a Python generator is and when to use it.",
            "What does this code do? x = [n**2 for n in range(10)]",
            "Summarize this page: https://www.python.org/about/",
        ],
    )

demo.launch()